In [1]:
pip install faiss-cpu

     |████████████████████████████████| 3.4 MB 5.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/mukesh/Development/project/health/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import faiss

cache = np.load(
    "../cache/entity_embeddings_cache.npz",
    allow_pickle=True,
)

embeddings = cache["embeddings"].astype("float32")


In [5]:
faiss.normalize_L2(embeddings)

In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(index.ntotal)

162212


In [9]:
faiss.write_index(
    index,
    "../cache/faiss.index"
)

In [10]:
metadata = {
    "names": cache["names"].tolist(),
    "cuis": cache["cuis"].tolist()
}

In [12]:
import pickle

with open(
    "../cache/faiss_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)

In [14]:
import faiss

index = faiss.read_index(
    "../cache/faiss.index"
)

In [21]:
import requests

FAISS_URL = "https://huggingface.co/datasets/testdevs/health_kg_embeddings/resolve/main/faiss.index"
METADATA_URL = "https://huggingface.co/datasets/testdevs/health_kg_embeddings/resolve/main/metadata.pkl"

def test_download(url):
    try:
        response = requests.get(url, stream=True, timeout=30)

        print(f"\nURL: {url}")
        print(f"Status Code: {response.status_code}")

        if response.status_code == 200:
            size_mb = int(response.headers.get("content-length", 0)) / (1024 * 1024)
            print(f"✓ File accessible")
            print(f"✓ Size: {size_mb:.2f} MB")
        else:
            print("✗ File not accessible")

    except Exception as e:
        print(f"✗ Error: {e}")

test_download(FAISS_URL)
test_download(METADATA_URL)


URL: https://huggingface.co/datasets/testdevs/health_kg_embeddings/resolve/main/faiss.index
Status Code: 200
✓ File accessible
✓ Size: 237.62 MB

URL: https://huggingface.co/datasets/testdevs/health_kg_embeddings/resolve/main/metadata.pkl
Status Code: 200
✓ File accessible
✓ Size: 5.05 MB


In [17]:
import time
start = time.time()

scores, ids = index.search(
    query_vector,
    10
)

print(time.time() - start)

NameError: name 'query_vector' is not defined